# Multi-Agent Research Assistant - Complete Jupyter Notebook

---
### 1: Install Required Libraries

In [ ]:
!pip install langchain langgraph langsmith langchain_openai langchain-community
!pip install faiss_cpu tavily-python pypdf python-dotenv openai tiktoken

print("Libraries installed!")

### 2: Import Libraries

In [ ]:
import os
import json
from typing import List, Dict, Optional, TypedDict
from datetime import datetime

# LangChain
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.schema import HumanMessage, SystemMessage
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS

# LangGraph
from langgraph.graph import StateGraph, END
from langgraph.checkpoint import MemorySaver

# Tavily for search
from tavily import TavilyClient

import warnings
warnings.filterwarnings('ignore')

print("Imports done!")

### 3: Setup API Keys

In [ ]:
OPENAI_API_KEY = "your-openai-api-key-here"
TAVILY_API_KEY = "your-tavily-api-key-here"
LANGCHAIN_API_KEY = "your-langchain-api-key-here"  # Optional

# Set environment variables
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY
os.environ["LANGCHAIN_API_KEY"] = LANGCHAIN_API_KEY
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Research_Assistant"

# Settings
MODEL_NAME = "gpt-3.5-turbo"  # "gpt-4"
TEMPERATURE = 0.7
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

print("API Keys configured!")


### 4: Define State

In [ ]:

class ResearchState(TypedDict):
    topic: str
    plan: List[str]
    search_results: List[Dict]
    retrieved_docs: List[str]
    notes: Dict[str, str]
    is_verified: bool
    missing_info: List[str]
    final_summary: str
    iterations: int

def create_initial_state(topic: str) -> ResearchState:
    return {
        "topic": topic,
        "plan": [],
        "search_results": [],
        "retrieved_docs": [],
        "notes": {},
        "is_verified": False,
        "missing_info": [],
        "final_summary": "",
        "iterations": 0
    }

print("State defined!")

### 5: Prompts

In [ ]:
PLANNER_PROMPT = """You are a research planner. Create a detailed research plan.

Topic: {topic}

Break down into 5-7 key sections.
Return only section names as a list.

Example for "Artificial Intelligence":
1. History and Evolution
2. Types of AI
3. Key Technologies
4. Current Applications
5. Challenges
6. Future Trends

Plan for: {topic}
"""

RESEARCH_PROMPT = """You are a research agent. Research this section.

Topic: {topic}
Section: {section}

Information from documents:
{documents}

Information from web search:
{search_results}

Write detailed notes with key facts and examples.
"""

VERIFIER_PROMPT = """You are a verification agent. Check research completeness.

Topic: {topic}
Research notes: {notes}

Answer:
1. Is research complete? (Yes/No)
2. What's missing? (List)
3. Is it reliable? (Yes/No)

Format:
COMPLETE: [Yes/No]
MISSING: [List]
RELIABLE: [Yes/No]
"""

SUMMARIZER_PROMPT = """You are a report writer. Create a professional report.

Topic: {topic}
Research Notes: {notes}

Write a comprehensive report with:
1. Introduction
2. Main sections
3. Key findings
4. Conclusion
5. References

Make it professional and well-structured.
"""

print("Prompts defined!")


### 6: Search Tool

In [ ]:

class SearchTool:
    def __init__(self):
        self.client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
    
    def search(self, query: str, max_results: int = 3) -> List[Dict]:
        try:
            results = self.client.search(
                query=query,
                max_results=max_results,
                include_raw_content=True
            )
            
            formatted = []
            for r in results.get('results', []):
                formatted.append({
                    'title': r.get('title', ''),
                    'content': r.get('content', ''),
                    'url': r.get('url', '')
                })
            return formatted
        except Exception as e:
            print(f"Search error: {e}")
            return []

search_tool = SearchTool()
print("Search tool ready!")


### 7: RAG Tool

In [ ]:

class RAGTool:
    def __init__(self):
        self.embeddings = OpenAIEmbeddings()
        self.vectorstore = None
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP
        )
    
    def load_documents(self, file_paths: List[str]):
        all_docs = []
        for path in file_paths:
            if path.endswith('.pdf'):
                loader = PyPDFLoader(path)
                all_docs.extend(loader.load())
            else:
                with open(path, 'r', encoding='utf-8') as f:
                    from langchain.schema import Document
                    all_docs.append(Document(page_content=f.read()))
        
        chunks = self.text_splitter.split_documents(all_docs)
        self.vectorstore = FAISS.from_documents(chunks, self.embeddings)
        print(f"Loaded {len(chunks)} chunks from {len(file_paths)} files")
    
    def retrieve(self, query: str, k: int = 3) -> List[str]:
        if not self.vectorstore:
            return []
        try:
            results = self.vectorstore.similarity_search(query, k=k)
            return [doc.page_content for doc in results]
        except:
            return []

rag_tool = RAGTool()
print("RAG tool ready!")


### 8: Create Agents

In [ ]:

class PlannerAgent:
    def __init__(self, llm):
        self.llm = llm
    
    def create_plan(self, topic: str) -> List[str]:
        prompt = PLANNER_PROMPT.format(topic=topic)
        response = self.llm.invoke([
            SystemMessage(content="You are a research planner."),
            HumanMessage(content=prompt)
        ])
        
        sections = []
        for line in response.content.strip().split('\n'):
            line = line.strip()
            if line and (line[0].isdigit() or line.startswith('-')):
                if line[0].isdigit():
                    section = line.split('.', 1)[-1].strip()
                else:
                    section = line[1:].strip()
                sections.append(section)
        return sections if sections else [response.content]

class ResearcherAgent:
    def __init__(self, llm):
        self.llm = llm
    
    def research_section(self, topic: str, section: str) -> str:
        # Search web
        search_results = search_tool.search(f"{topic} {section}")
        search_text = "\n".join([f"- {r['title']}: {r['content'][:300]}" for r in search_results])
        
        # RAG retrieval
        docs = rag_tool.retrieve(f"{topic} {section}")
        doc_text = "\n".join([f"Doc: {d[:300]}" for d in docs])
        
        prompt = RESEARCH_PROMPT.format(
            topic=topic,
            section=section,
            documents=doc_text or "No documents available",
            search_results=search_text or "No search results"
        )
        
        response = self.llm.invoke([
            SystemMessage(content="You are a research assistant."),
            HumanMessage(content=prompt)
        ])
        return response.content
    
    def research_all(self, topic: str, plan: List[str]) -> Dict[str, str]:
        notes = {}
        for section in plan:
            print(f"  Researching: {section}")
            notes[section] = self.research_section(topic, section)
        return notes

class VerifierAgent:
    def __init__(self, llm):
        self.llm = llm
    
    def verify(self, topic: str, notes: Dict[str, str]) -> tuple:
        notes_text = "\n".join([f"=== {s} ===\n{c}" for s, c in notes.items()])
        
        prompt = VERIFIER_PROMPT.format(topic=topic, notes=notes_text)
        response = self.llm.invoke([
            SystemMessage(content="You are a verification agent."),
            HumanMessage(content=prompt)
        ])
        
        text = response.content
        is_complete = "COMPLETE: Yes" in text
        is_reliable = "RELIABLE: Yes" in text
        
        missing = []
        if "MISSING:" in text:
            missing_part = text.split("MISSING:")[1].split("RELIABLE:")[0].strip()
            if missing_part and missing_part.lower() != "none":
                missing = [m.strip() for m in missing_part.split('\n') if m.strip()]
        
        return is_complete, missing, is_reliable

class SummarizerAgent:
    def __init__(self, llm):
        self.llm = llm
    
    def summarize(self, topic: str, notes: Dict[str, str]) -> str:
        notes_text = "\n".join([f"=== {s} ===\n{c}" for s, c in notes.items()])
        
        prompt = SUMMARIZER_PROMPT.format(topic=topic, notes=notes_text)
        response = self.llm.invoke([
            SystemMessage(content="You are a report writer."),
            HumanMessage(content=prompt)
        ])
        return response.content

print("All agents created!")


### 9: Build LangGraph Workflow

In [ ]:

class ResearchAssistant:
    def __init__(self):
        self.llm = ChatOpenAI(model=MODEL_NAME, temperature=TEMPERATURE)
        
        self.planner = PlannerAgent(self.llm)
        self.researcher = ResearcherAgent(self.llm)
        self.verifier = VerifierAgent(self.llm)
        self.summarizer = SummarizerAgent(self.llm)
        
        self.workflow = self._build_workflow()
        self.memory = MemorySaver()
        self.app = self.workflow.compile(checkpointer=self.memory)
    
    def _build_workflow(self):
        workflow = StateGraph(ResearchState)
        
        workflow.add_node("planner", self._plan_node)
        workflow.add_node("researcher", self._research_node)
        workflow.add_node("verifier", self._verify_node)
        workflow.add_node("summarizer", self._summary_node)
        
        workflow.set_entry_point("planner")
        workflow.add_edge("planner", "researcher")
        workflow.add_edge("researcher", "verifier")
        
        workflow.add_conditional_edges(
            "verifier",
            self._decide_next,
            {
                "continue_research": "researcher",
                "finish": "summarizer"
            }
        )
        
        workflow.add_edge("summarizer", END)
        return workflow
    
    def _plan_node(self, state: ResearchState) -> dict:
        print("Creating research plan...")
        plan = self.planner.create_plan(state["topic"])
        return {"plan": plan}
    
    def _research_node(self, state: ResearchState) -> dict:
        print("Researching...")
        plan = state.get("plan", [])
        if not plan:
            plan = self.planner.create_plan(state["topic"])
        
        notes = self.researcher.research_all(state["topic"], plan)
        iterations = state.get("iterations", 0) + 1
        
        return {"notes": notes, "iterations": iterations}
    
    def _verify_node(self, state: ResearchState) -> dict:
        print("Verifying research...")
        is_complete, missing, is_reliable = self.verifier.verify(
            state["topic"],
            state.get("notes", {})
        )
        return {
            "is_verified": is_complete and is_reliable,
            "missing_info": missing
        }
    
    def _summary_node(self, state: ResearchState) -> dict:
        print("Generating report...")
        summary = self.summarizer.summarize(
            state["topic"],
            state.get("notes", {})
        )
        return {"final_summary": summary}
    
    def _decide_next(self, state: ResearchState) -> str:
        is_verified = state.get("is_verified", False)
        iterations = state.get("iterations", 0)
        
        if not is_verified and iterations < 3:
            print(f"Missing: {state.get('missing_info', [])}")
            return "continue_research"
        return "finish"
    
    def research(self, topic: str) -> dict:
        print(f"\n{'='*50}")
        print(fResearching: {topic}")
        print(f"{'='*50}\n")
        
        initial_state = create_initial_state(topic)
        config = {"configurable": {"thread_id": "research"}}
        
        final_state = self.app.invoke(initial_state, config)
        
        return {
            "topic": topic,
            "plan": final_state.get("plan", []),
            "notes": final_state.get("notes", {}),
            "report": final_state.get("final_summary", ""),
            "iterations": final_state.get("iterations", 0),
            "verified": final_state.get("is_verified", False)
        }

print("Research Assistant ready!")


### 10: Test with Sample Documents (Optional)

In [ ]:
# sample_docs = ["documents/sample.pdf"]  # PDF path
# rag_tool.load_documents(sample_docs)

# create sample text document ा
sample_text = """
Artificial Intelligence (AI) is the simulation of human intelligence in machines.
Machine Learning is a subset of AI that enables systems to learn from data.
Deep Learning uses neural networks with many layers.
Natural Language Processing helps computers understand human language.
Computer Vision enables machines to interpret visual information.
"""

# Temporary file create
with open("sample_doc.txt", "w") as f:
    f.write(sample_text)

# Load the sample document
rag_tool.load_documents(["sample_doc.txt"])
print("Sample document loaded!")


### 11: Run Research Assistant

In [ ]:
# Create assistant
assistant = ResearchAssistant()

# Run research
topic = "Artificial General Intelligence" ा
result = assistant.research(topic)

# Display results
print(f"RESEARCH PLAN")
for i, section in enumerate(result["plan"], 1):
    print(f"{i}. {section}")

print(f"\n{'='*50}")
print(f"FINAL REPORT")
print(result["report"])

print(f"\n{'='*50}")
print(f"STATISTICS")
print(f"Topic: {result['topic']}")
print(f"Sections: {len(result['plan'])}")
print(f"Iterations: {result['iterations']}")
print(f"Verified: {'Yes' if result['verified'] else 'Partial'}")


### 12: Save Report to File

In [ ]:
filename = f"{topic.replace(' ', '_')}_report.txt"
with open(filename, "w", encoding="utf-8") as f:
    f.write(f"{'='*60}\n")
    f.write(f"RESEARCH REPORT: {topic}\n")
    f.write(f"{'='*60}\n\n")
    f.write(result["report"])
    f.write(f"\n\n{'='*60}\n")
    f.write(f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Iterations: {result['iterations']}\n")
    f.write(f"Verified: {result['verified']}\n")

print(f"Report saved to: {filename}")

### 13: Interactive Research (Optional)

In [ ]:
def interactive_research():
    print("MULTI-AGENT RESEARCH ASSISTANT")
    
    while True:
        topic = input("\nEnter research topic (or 'quit' to exit): ").strip()
        
        if topic.lower() == 'quit':
            print("Goodbye!")
            break
        
        if not topic:
            print("Please enter a topic.")
            continue
        
        try:
            result = assistant.research(topic)
            
            print("PLAN")
            for i, s in enumerate(result["plan"], 1):
                print(f"{i}. {s}")
            
            print("REPORT")
            print(result["report"][:1000] + "..." if len(result["report"]) > 1000 else result["report"])
            
            # Save
            filename = f"{topic.replace(' ', '_')}_report.txt"
            with open(filename, "w", encoding="utf-8") as f:
                f.write(result["report"])
            print(f"\nFull report saved to: {filename}")
            
        except Exception as e:
            print(f"Error: {e}")

# Run interactive mode
# interactive_research() 

### 14: Test with Different Topics

In [ ]:
test_topics = [
    "Machine Learning",
    "Climate Change",
    "Quantum Computing"
]

for topic in test_topics:
    print(f"Testing: {topic}")
    
    try:
        result = assistant.research(topic)
        print(f"Done! Sections: {len(result['plan'])}, Iterations: {result['iterations']}")
        print(f"Report preview: {result['report'][:200]}...")
    except Exception as e:
        print(f"Error: {e}")

### 15: Advanced - Custom Research

In [ ]:
def custom_research(topic: str, max_iterations: int = 3, verbose: bool = True):
    """
    Custom research function
    
    Args:
        topic: Research topic
        max_iterations: Maximum research iterations
        verbose: Print progress or not
    """
    
    # Create custom assistant with different settings
    custom_llm = ChatOpenAI(
        model="gpt-3.5-turbo",
        temperature=0.5 
    )
    
    # Create agents with custom LLM
    planner = PlannerAgent(custom_llm)
    researcher = ResearcherAgent(custom_llm)
    verifier = VerifierAgent(custom_llm)
    summarizer = SummarizerAgent(custom_llm)
    
    # Research process
    if verbose:
        print(f"\nResearching: {topic}")
    
    # Step 1: Plan
    if verbose:
        print("Creating plan...")
    plan = planner.create_plan(topic)
    
    # Step 2: Research
    if verbose:
        print("Gathering information...")
    notes = researcher.research_all(topic, plan)
    
    # Step 3: Verify
    if verbose:
        print("Verifying...")
    is_complete, missing, is_reliable = verifier.verify(topic, notes)
    
    # Self-correction loop
    iteration = 1
    while not is_complete and iteration < max_iterations:
        if verbose:
            print(f"Iteration {iteration + 1}: Researching missing info...")
        
        # Research missing sections
        for section in missing:
            notes[section] = researcher.research_section(topic, section)
        
        # Verify again
        is_complete, missing, is_reliable = verifier.verify(topic, notes)
        iteration += 1
    
    # Step 4: Summarize
    if verbose:
        print("Generating report...")
    report = summarizer.summarize(topic, notes)
    
    return {
        "topic": topic,
        "plan": plan,
        "notes": notes,
        "report": report,
        "iterations": iteration,
        "verified": is_complete and is_reliable
    }

# Test custom research
result = custom_research("Renewable Energy", max_iterations=2)
print(f"\nResearch complete!")
print(f"Iterations: {result['iterations']}")
print(f"Verified: {result['verified']}")
print(f"\nReport preview:\n{result['report'][:500]}...")


### 16: Utility Functions

In [ ]:
def extract_key_points(report: str, num_points: int = 5) -> List[str]:
    """Report : key points extract"""
    llm = ChatOpenAI(model=MODEL_NAME, temperature=0.3)
    
    prompt = f"""Extract {num_points} key points from this report:

{report}

Return only the key points as a list.
"""
    
    response = llm.invoke([
        SystemMessage(content="You are a key point extractor."),
        HumanMessage(content=prompt)
    ])
    
    points = []
    for line in response.content.strip().split('\n'):
        line = line.strip()
        if line and (line[0].isdigit() or line.startswith('-') or line.startswith('•')):
            if line[0].isdigit():
                point = line.split('.', 1)[-1].strip()
            else:
                point = line[1:].strip()
            points.append(point)
    
    return points if points else [response.content]

def get_report_summary(report: str, max_words: int = 100) -> str:
    """Report short summary"""
    llm = ChatOpenAI(model=MODEL_NAME, temperature=0.3)
    
    prompt = f"""Summarize this report in {max_words} words:

{report}

Summary:"""
    
    response = llm.invoke([
        SystemMessage(content="You are a summarizer."),
        HumanMessage(content=prompt)
    ])
    
    return response.content

# Test utility functions
if 'result' in locals():
    print("\nKey Points:")
    points = extract_key_points(result['report'])
    for i, point in enumerate(points, 1):
        print(f"{i}. {point}")
    
    print(f"\nSummary:")
    print(get_report_summary(result['report'], 50))

print("\nAll utility functions ready!")

### 17: Batch Research

In [ ]:
def batch_research(topics: List[str], save_reports: bool = True) -> Dict[str, dict]:
    """
    Multiple topics research 
    
    Args:
        topics: List of topics
        save_reports: Save reports to files or not
    
    Returns:
        Dictionary of results
    """
    results = {}
    
    print(f"BATCH RESEARCH - {len(topics)} Topics")
    
    for i, topic in enumerate(topics, 1):
        print(f"\n[{i}/{len(topics)}] Researching: {topic}")
        
        try:
            result = assistant.research(topic)
            results[topic] = result
            
            if save_reports:
                filename = f"{topic.replace(' ', '_')}_report.txt"
                with open(filename, "w", encoding="utf-8") as f:
                    f.write(result["report"])
                print(f"Saved to: {filename}")
            
        except Exception as e:
            print(f"Error: {e}")
            results[topic] = {"error": str(e)}
    
    print(f"Batch research complete!")
    
    return results

# Run batch research
batch_topics = ["Deep Learning", "Natural Language Processing"]
batch_results = batch_research(batch_topics)
